In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Repository Clone
%cd /kaggle/working
!git clone https://github.com/LakshayBaijal/Computer-Vision_Assignments_Lakshay.git
%cd /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1

/kaggle/working
Cloning into 'Computer-Vision_Assignments_Lakshay'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 30 (delta 1), reused 27 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 344.64 KiB | 3.45 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/Computer-Vision_Assignments_Lakshay/Q1


In [7]:
from pathlib import Path

# Q1 repo path
repo_candidates = list(Path("/kaggle/working").rglob("Q1/train.py"))
assert repo_candidates, "Q1/train.py not found in /kaggle/working"
REPO = repo_candidates[0].parent
print("REPO:", REPO)

# Dataset path
data_root = None
for d in Path("/kaggle/input").rglob("*"):
    if d.is_dir() and (d / "img").exists() and (d / "annots").exists():
        data_root = d
        break
assert data_root is not None, "Dataset root (img + annots) not found in /kaggle/input"
print("DATA_ROOT:", data_root)

REPO: /kaggle/working/Computer-Vision_Assignments_Lakshay/Q1
DATA_ROOT: /kaggle/input/datasets/lakshaybaijal/q1-dataset


In [4]:
%pip install -q einops matplotlib "numpy>=1.26,<2.2" opencv-python PyYAML tqdm shapely Pillow

Note: you may need to restart the kernel to use updated packages.


In [8]:
import yaml
from pathlib import Path

base_cfg = REPO / "config" / "st.yaml"
run2_cfg = Path("/kaggle/working/run2_aabb_hp2.yaml")

with open(base_cfg, "r") as f:
    cfg = yaml.safe_load(f)

cfg["dataset_params"]["root_dir"] = str(data_root)

cfg["train_params"]["use_angle"] = False
cfg["train_params"]["num_epochs"] = 10
cfg["train_params"]["task_name"] = "/kaggle/working/Q1_run2_aabb_hp2_ckpts"
cfg["train_params"]["ckpt_name"] = "aabb_hp2.pth"

# HP2 change (different from Run1)
cfg["train_params"]["lr"] = 5e-4
cfg["train_params"]["lr_steps"] = [7, 9]

with open(run2_cfg, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Saved:", run2_cfg)
print(cfg["train_params"])

Saved: /kaggle/working/run2_aabb_hp2.yaml
{'task_name': '/kaggle/working/Q1_run2_aabb_hp2_ckpts', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 10, 'lr_steps': [7, 9], 'lr': 0.0005, 'ckpt_name': 'aabb_hp2.pth', 'use_angle': False}


In [9]:
import os, sys
os.chdir(str(REPO))
sys.path.insert(0, str(REPO))

from train import train

class Args:
    config_path = "/kaggle/working/run2_aabb_hp2.yaml"
    root_dir = str(data_root)

train(Args())

Using device: cuda
{'dataset_params': {'root_dir': '/kaggle/input/datasets/lakshaybaijal/q1-dataset', 'num_classes': 2}, 'model_params': {'im_channels': 3, 'aspect_ratios': [0.5, 1, 2], 'scales': [128, 256, 512], 'min_im_size': 600, 'max_im_size': 1000, 'backbone_out_channels': 512, 'fc_inner_dim': 1024, 'rpn_bg_threshold': 0.3, 'rpn_fg_threshold': 0.7, 'rpn_nms_threshold': 0.7, 'rpn_train_prenms_topk': 12000, 'rpn_test_prenms_topk': 6000, 'rpn_train_topk': 2000, 'rpn_test_topk': 300, 'rpn_batch_size': 256, 'rpn_pos_fraction': 0.5, 'roi_iou_threshold': 0.5, 'roi_low_bg_iou': 0.0, 'roi_pool_size': 7, 'roi_nms_threshold': 0.3, 'roi_topk_detections': 100, 'roi_score_threshold': 0.05, 'roi_batch_size': 128, 'roi_pos_fraction': 0.25, 'bbox_reg_weights': [10.0, 10.0, 5.0, 5.0, 1.0]}, 'train_params': {'task_name': '/kaggle/working/Q1_run2_aabb_hp2_ckpts', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 10, 'lr_steps': [7, 9], 'lr': 0.0005, 'ckpt_name': 'aabb_hp2.pth', 'use_ang

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 207MB/s] 


Using angle prediction: False
Checkpoints will be saved to: /kaggle/working/Q1_run2_aabb_hp2_ckpts
Effective learning rate: 0.0005


...

Finished epoch 7
Checkpoint saved: /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth
RPN Classification Loss : 0.0264 | RPN Localization Loss : 0.0600 | FRCNN Classification Loss : 0.5584 | FRCNN Localization Loss : 0.2997


100%|██████████| 181/181 [02:37<00:00,  1.15it/s]


Finished epoch 8
Checkpoint saved: /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth
RPN Classification Loss : 0.0241 | RPN Localization Loss : 0.0571 | FRCNN Classification Loss : 0.5401 | FRCNN Localization Loss : 0.2926


100%|██████████| 181/181 [02:38<00:00,  1.14it/s]


Finished epoch 9
Checkpoint saved: /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth
RPN Classification Loss : 0.0216 | RPN Localization Loss : 0.0542 | FRCNN Classification Loss : 0.5192 | FRCNN Localization Loss : 0.2850
Done Training...


In [10]:
import os, glob, json, shutil
from IPython.display import FileLink, display, HTML

ckpt_dir = "/kaggle/working/Q1_run2_aabb_hp2_ckpts"
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.pth")))
assert ckpts, f"No checkpoints found in {ckpt_dir}"

export_dir = "/kaggle/working/EXPORT_Q1_RUN2_AABB_HP2"
os.makedirs(export_dir, exist_ok=True)

latest = ckpts[-1]
shutil.copy2(latest, os.path.join(export_dir, "Q1_RUN2_AABB_HP2_final.pth"))
with open(os.path.join(export_dir, "Q1_RUN2_AABB_HP2_manifest.json"), "w") as f:
    json.dump({"latest": latest, "all_checkpoints": ckpts}, f, indent=2)

zip_path = shutil.make_archive("/kaggle/working/Q1_RUN2_AABB_HP2_EXPORT", "zip", export_dir)
print("ZIP:", zip_path)

display(FileLink("/kaggle/working/Q1_RUN2_AABB_HP2_EXPORT.zip"))
display(HTML('<script>window.open("/files/Q1_RUN2_AABB_HP2_EXPORT.zip","_blank");</script>'))
print("If blocked, click the link above.")

ZIP: /kaggle/working/Q1_RUN2_AABB_HP2_EXPORT.zip


/kaggle/working/Q1_RUN2_AABB_HP2_EXPORT.zip

If blocked, click the link above.


In [11]:
import os, glob, shutil, json, time
from IPython.display import display, FileLink, HTML

# 1) Debug: list likely checkpoint files
all_pth = sorted(glob.glob("/kaggle/working/**/*.pth", recursive=True))
print("Found .pth files:", len(all_pth))
for p in all_pth[-20:]:
    print(" -", p)

# 2) Pick Run2-like files first, else fallback to latest .pth
run2_candidates = [p for p in all_pth if "run2" in p.lower() or "hp2" in p.lower() or "aabb_hp2" in p.lower()]
if run2_candidates:
    chosen = run2_candidates[-1]
else:
    assert all_pth, "No .pth found in /kaggle/working. Train must finish first."
    chosen = all_pth[-1]

print("\nUsing checkpoint:", chosen)

# 3) Build fresh export folder + zip (new unique name avoids stale links)
tag = time.strftime("%Y%m%d_%H%M%S")
export_dir = f"/kaggle/working/EXPORT_Q1_RUN2_AABB_HP2_{tag}"
os.makedirs(export_dir, exist_ok=True)

final_pth = os.path.join(export_dir, "Q1_RUN2_AABB_HP2_final.pth")
shutil.copy2(chosen, final_pth)

manifest = {
    "selected_checkpoint": chosen,
    "exported_final_model": final_pth
}
with open(os.path.join(export_dir, "Q1_RUN2_AABB_HP2_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

zip_base = f"/kaggle/working/Q1_RUN2_AABB_HP2_EXPORT_{tag}"
zip_path = shutil.make_archive(zip_base, "zip", export_dir)
zip_name = os.path.basename(zip_path)

print("\nCreated zip:", zip_path)
print("Size (MB):", os.path.getsize(zip_path)/1024/1024)

# 4) Valid links
display(FileLink(zip_path))
display(HTML(f'<a href="/files/{zip_name}" target="_blank">Download {zip_name}</a>'))
display(HTML(f'<script>window.open("/files/{zip_name}", "_blank");</script>'))
print("If browser blocks popup, click the link above manually.")

Found .pth files: 2
 - /kaggle/working/EXPORT_Q1_RUN2_AABB_HP2/Q1_RUN2_AABB_HP2_final.pth
 - /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth

Using checkpoint: /kaggle/working/Q1_run2_aabb_hp2_ckpts/tv_frcnn_r50fpn_aabb_hp2.pth

Created zip: /kaggle/working/Q1_RUN2_AABB_HP2_EXPORT_20260405_121149.zip
Size (MB): 146.8490390777588


/kaggle/working/Q1_RUN2_AABB_HP2_EXPORT_20260405_121149.zip

If browser blocks popup, click the link above manually.
